In [65]:
import pandas as pd
import numpy as np
import ast
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [66]:
# Load datasets
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print(f"Movies shape: {movies.shape}")
print(f"Credits shape: {credits.shape}")
print("✓ Datasets loaded successfully")

Movies shape: (4803, 20)
Credits shape: (4803, 4)
✓ Datasets loaded successfully


In [67]:
# Merge on title
movies = movies.merge(credits, on='title')

print(f"Merged shape: {movies.shape}")
print("\nColumn names after merge:")
print(movies.columns.tolist())
print("\n✓ Datasets merged successfully")

movies.head(2)

Merged shape: (4809, 23)

Column names after merge:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'movie_id', 'cast', 'crew']

✓ Datasets merged successfully


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [68]:
# Keep only necessary columns
# Note: Use 'id' instead of 'movie_id' (movie_id comes from credits dataset)
movies = movies[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Rename 'id' to 'movie_id' for clarity
movies.rename(columns={'id': 'movie_id'}, inplace=True)

print(f"Selected columns: {movies.shape}")
print(f"Column names: {movies.columns.tolist()}")
print("✓ Columns selected")

movies.head(2)

Selected columns: (4809, 7)
Column names: ['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']
✓ Columns selected


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [69]:
# Check missing values
print("Missing values before cleaning:")
print(movies.isnull().sum())
print()

# Drop rows with missing values
movies.dropna(inplace=True)

print(f"Shape after dropping NaN: {movies.shape}")
print(f"Duplicates: {movies.duplicated().sum()}")
print("✓ Missing values handled")

Missing values before cleaning:
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

Shape after dropping NaN: (4806, 7)
Duplicates: 0
✓ Missing values handled


In [70]:
def convert(obj):
    """Convert string representation of list to actual list of names"""
    L = []
    
    # Handle if already a list
    if isinstance(obj, list):
        for i in obj:
            if isinstance(i, dict) and 'name' in i:
                L.append(i['name'])
        return L
    
    # Handle string representation
    if isinstance(obj, str):
        try:
            parsed = ast.literal_eval(obj)
            for i in parsed:
                if isinstance(i, dict) and 'name' in i:
                    L.append(i['name'])
        except (ValueError, SyntaxError):
            # If parsing fails, return empty list
            pass
    
    return L


def convert_cast(obj):
    """Get top 3 cast members"""
    L = []
    counter = 0
    
    # Handle if already a list
    if isinstance(obj, list):
        for i in obj:
            if counter < 3 and isinstance(i, dict) and 'name' in i:
                L.append(i['name'])
                counter += 1
        return L
    
    # Handle string representation
    if isinstance(obj, str):
        try:
            parsed = ast.literal_eval(obj)
            for i in parsed:
                if counter < 3 and isinstance(i, dict) and 'name' in i:
                    L.append(i['name'])
                    counter += 1
        except (ValueError, SyntaxError):
            pass
    
    return L


def fetch_director(obj):
    """Extract director name from crew"""
    L = []
    
    # Handle if already a list
    if isinstance(obj, list):
        for i in obj:
            if isinstance(i, dict) and i.get('job') == 'Director':
                L.append(i['name'])
                break
        return L
    
    # Handle string representation
    if isinstance(obj, str):
        try:
            parsed = ast.literal_eval(obj)
            for i in parsed:
                if isinstance(i, dict) and i.get('job') == 'Director':
                    L.append(i['name'])
                    break
        except (ValueError, SyntaxError):
            pass
    
    return L


print("✓ Helper functions defined (with error handling)")

✓ Helper functions defined (with error handling)


In [71]:
# Apply conversions with progress feedback
print("Applying transformations...")

try:
    print("  → Processing genres...")
    movies['genres'] = movies['genres'].apply(convert)
    
    print("  → Processing keywords...")
    movies['keywords'] = movies['keywords'].apply(convert)
    
    print("  → Processing cast...")
    movies['cast'] = movies['cast'].apply(convert_cast)
    
    print("  → Processing crew...")
    movies['crew'] = movies['crew'].apply(fetch_director)
    
    print("✓ All transformations applied successfully")
    
except Exception as e:
    print(f"❌ Error during transformation: {e}")
    print("\nShowing problematic row:")
    print(movies.head(1))

movies.head(2)

Applying transformations...
  → Processing genres...
  → Processing keywords...
  → Processing cast...
  → Processing crew...
✓ All transformations applied successfully


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]


In [72]:
# Convert overview to list of words
# Handle NaN and ensure it's a list
def process_overview(text):
    if pd.isna(text):
        return []
    if isinstance(text, list):
        return text
    if isinstance(text, str):
        return text.split()
    return []

movies['overview'] = movies['overview'].apply(process_overview)

print("✓ Overview processed")
print(f"Sample overview: {movies['overview'].iloc[0][:5]}...")  # Show first 5 words

movies.head(2)

✓ Overview processed
Sample overview: ['In', 'the', '22nd', 'century,', 'a']...


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]


In [73]:
# Remove spaces (e.g., "Science Fiction" -> "ScienceFiction")
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

print("✓ Spaces removed from names")

movies.head(2)

✓ Spaces removed from names


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]


In [74]:
# Combine all features into tags
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

print("✓ Tags created")

movies[['title', 'tags']].head(2)

✓ Tags created


,title,tags
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."


In [75]:
# Keep only necessary columns
new_df = movies[['movie_id', 'title', 'tags']].copy()

print(f"New dataframe shape: {new_df.shape}")
print("✓ New dataframe created")

new_df.head()

New dataframe shape: (4806, 3)
✓ New dataframe created


,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [76]:
# Join list of tags into single string
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

print("✓ Tags converted to string")

new_df.head(2)

✓ Tags converted to string


,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."


In [77]:
# Lowercase for better matching
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

print("✓ Tags converted to lowercase")

new_df.head(2)

✓ Tags converted to lowercase


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."


In [78]:
# Lowercase for better matching
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

print("✓ Tags converted to lowercase")

new_df.head(2)

✓ Tags converted to lowercase


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."


In [79]:
# Safety check
empty_tags = new_df[new_df['tags'].str.len() == 0]
print(f"Movies with empty tags: {len(empty_tags)}")

if len(empty_tags) > 0:
    print("Removing movies with empty tags...")
    new_df = new_df[new_df['tags'].str.len() > 0].copy()

print(f"Final dataframe shape: {new_df.shape}")
print("✓ Empty tags handled")

Movies with empty tags: 0
Final dataframe shape: (4806, 3)
✓ Empty tags handled


In [80]:
# Create CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

# Transform tags into vectors
vectors = cv.fit_transform(new_df['tags']).toarray()

print(f"Vectors shape: {vectors.shape}")
print(f"Vocabulary size: {len(cv.vocabulary_)}")
print("✓ Vectorization complete")

Vectors shape: (4806, 5000)
Vocabulary size: 5000
✓ Vectorization complete


In [81]:
# Calculate cosine similarity
similarity = cosine_similarity(vectors)

print(f"Similarity matrix shape: {similarity.shape}")
print(f"Matrix size in memory: ~{similarity.nbytes / (1024*1024):.2f} MB")
print("✓ Similarity matrix computed")

Similarity matrix shape: (4806, 4806)
Matrix size in memory: ~176.22 MB
✓ Similarity matrix computed


In [82]:
def recommend(movie):
    """Recommend 5 similar movies"""
    # Find movie index
    movie_index = new_df[new_df['title'] == movie].index[0]
    
    # Get similarity scores
    distances = similarity[movie_index]
    
    # Sort and get top 5 (excluding itself)
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"\nMovies similar to '{movie}':\n")
    for i in movies_list:
        print(f"  • {new_df.iloc[i[0]].title}")

# Test with different movies
recommend('Avatar')


Movies similar to 'Avatar':

  • Titan A.E.
  • Small Soldiers
  • Independence Day
  • Ender's Game
  • Aliens vs Predator: Requiem


In [83]:
# Test with more examples
test_movies = ['The Dark Knight', 'Titanic', 'Toy Story']

for movie in test_movies:
    try:
        recommend(movie)
    except IndexError:
        print(f"\n'{movie}' not found in database")


Movies similar to 'The Dark Knight':

  • The Dark Knight Rises
  • Batman Begins
  • Batman Returns
  • Batman Forever
  • Batman & Robin

Movies similar to 'Titanic':

  • Raise the Titanic
  • Captain Phillips
  • The Notebook
  • In the Heart of the Sea
  • Ghost Ship

Movies similar to 'Toy Story':

  • Toy Story 2
  • Toy Story 3
  • The 40 Year Old Virgin
  • Heartbeeps
  • Max Keeble's Big Move


In [84]:
# Save processed data
print("Saving pickle files...")

# Save movies dictionary
pickle.dump(new_df.to_dict(), open('movies_dict.pkl', 'wb'))
print("✓ movies_dict.pkl saved")

# Save similarity matrix
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print("✓ similarity.pkl saved")

print("\n🎉 All files saved successfully!")
print("\nGenerated files:")
print("  • movies_dict.pkl")
print("  • similarity.pkl")

Saving pickle files...
✓ movies_dict.pkl saved
✓ similarity.pkl saved

🎉 All files saved successfully!

Generated files:
  • movies_dict.pkl
  • similarity.pkl


In [85]:
import os

# Check if files exist and their sizes
files = ['movies_dict.pkl', 'similarity.pkl']

print("File verification:")
for file in files:
    if os.path.exists(file):
        size_mb = os.path.getsize(file) / (1024 * 1024)
        print(f"✓ {file} - {size_mb:.2f} MB")
    else:
        print(f"✗ {file} - NOT FOUND")

File verification:
✓ movies_dict.pkl - 2.26 MB
✓ similarity.pkl - 176.22 MB
